# 02 · Colour features, PCA and K-means

Second stage of the exploration.
After the brightness-only models ([notebook 01](01_grayscale_threshold_morphology.ipynb)) I looked at the **colour**
of each pixel, checked it with a PCA, and let **K-means** find the groups by itself.

**To run it** you need the scans in `data/images/` as `vir_<id>.jpg`. The last section (model 1 against model 2) runs
without images, from `data/standardization_comparison_50_leaves.csv`. The scans and the tables in `data/` are not
included in the repository and are available on reasonable request.

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import decomposition, preprocessing
from sklearn.cluster import KMeans

IMAGES_DIR = Path("../../data/images")
REFERENCE = (pd.read_csv("../../data/pliman_validation_50_leaves.csv")
               .set_index("fig_id")["pliman_severity"])

RGB = ["red", "blue", "green"]
DIFFS = ["diff_r_g", "diff_r_b", "diff_b_g"]


def read_rgb(leaf_id):
    img = cv2.imread(str(IMAGES_DIR / f"vir_{leaf_id}.jpg"))
    if img is None:
        raise FileNotFoundError(f"vir_{leaf_id}.jpg not found in {IMAGES_DIR}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def pixel_table(img_rgb):
    """One row per pixel: position, colour and the differences between colour channels (as int, so they do not overflow)."""
    rows, cols = img_rgb.shape[:2]
    r, c = np.indices((rows, cols))
    px = img_rgb.astype(int)
    df = pd.DataFrame({"row": r.ravel(), "col": c.ravel(),
                       "red": px[..., 0].ravel(), "green": px[..., 1].ravel(), "blue": px[..., 2].ravel()})
    df["diff_r_g"] = df["red"] - df["green"]
    df["diff_r_b"] = df["red"] - df["blue"]
    df["diff_b_g"] = df["blue"] - df["green"]
    return df

## 1. Look at the colour channels

Same lesion and healthy patches as before, now one histogram per channel. Inside each patch the three channels go up
and down together: most of what the three carry is brightness.

In [ ]:
leaf46 = read_rgb(46)
lesion = leaf46[60:120, 160:220]
healthy = leaf46[310:370, 170:230]

fig, ax = plt.subplots(2, 3, figsize=(14, 7))
for row, (patch, name) in enumerate([(lesion, "Lesion"), (healthy, "Healthy tissue")]):
    for k, (channel, color) in enumerate([("red", "red"), ("green", "green"), ("blue", "blue")]):
        ax[row, k].hist(patch[..., k].ravel(), bins=32, range=(0, 256), color=color, alpha=0.6)
        ax[row, k].set(title=f"{name}: {channel}", xlabel="Brightness", ylabel="Pixels")
plt.tight_layout(); plt.show()

## 2. Reduce the noise

Median blur, kernel 5 at first and 11 from the PCA onwards (11 is what every later model uses).

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, (img, title) in zip(ax, [(leaf46, "Original"),
                                (cv2.medianBlur(leaf46, 5), "Median blur 5"),
                                (cv2.medianBlur(leaf46, 11), "Median blur 11")]):
    a.imshow(img); a.set_title(title); a.axis("off")
plt.show()

## 3. The pixels in colour space, and the differences between channels

Every pixel is a point (red, green, blue). I added the **differences** red − green, red − blue and blue − green
to see if they separated the lesions better than brightness alone.

In [ ]:
df46 = pixel_table(leaf46)

fig = plt.figure(figsize=(12, 5))
ax = fig.add_subplot(121, projection="3d")
ax.scatter(df46["red"], df46["green"], df46["blue"], c="gray", s=1)
ax.scatter(0, 0, 0, c="black", s=25); ax.scatter(255, 255, 255, c="red", s=25)
ax.set(xlabel="red", ylabel="green", zlabel="blue")

ax2 = fig.add_subplot(122)
ax2.scatter(df46["diff_r_g"], df46["diff_b_g"], c="gray", s=1)
ax2.set(xlabel="red - green", ylabel="blue - green")
plt.show()

## 4. PCA: how much information is in the three colours?

PCA on the standardised features, with and without the differences, and with and without median blur (kernel 11).
The component images are what I looked at. With the three colour channels alone, the first component carries
**99.8 %** of the variance on leaf 46 (99.84 % without blur, 99.87 % with it): the three channels are almost the same
information.

What the component images showed:

* PCA works, and a priori it solves the problem of the vein.
* Component 2 with blur, for colour + differences, finds the lesions well but does not find the surface of the leaf.
* Component 1 with blur and **without** differences finds the lesions and also separates the leaf from the background.
* Component 1 with differences finds the leaf but not the lesions.

In [ ]:
def run_pca(table, features, n_components=3):
    scaled = preprocessing.StandardScaler().fit_transform(table[features])
    pca = decomposition.PCA(n_components=n_components).fit(scaled)
    return pca, pca.transform(scaled)


def show_components(img_rgb, features, title):
    table = pixel_table(img_rgb)
    pca, projected = run_pca(table, features)
    print(f"{title}: explained variance per component = {np.round(pca.explained_variance_ratio_, 4)}")
    rows, cols = img_rgb.shape[:2]
    fig, ax = plt.subplots(1, 3, figsize=(13, 5))
    for k in range(3):
        im = ax[k].imshow(projected[:, k].reshape(rows, cols), cmap="inferno")
        ax[k].set_title(f"{title} - component {k + 1}"); ax[k].axis("off"); fig.colorbar(im, ax=ax[k], shrink=0.5)
    plt.show()


blurred46 = cv2.medianBlur(leaf46, 11)
show_components(leaf46,    RGB,          "RGB")
show_components(leaf46,    RGB + DIFFS,  "RGB + differences")
show_components(blurred46, RGB,          "RGB, blur 11")
show_components(blurred46, RGB + DIFFS,  "RGB + differences, blur 11")

## 5. K-means with three groups

Instead of picking components by hand, I gave the pixels to K-means with three groups (background, healthy tissue,
lesion), with colour alone and with colour plus differences, on a leaf with few lesions (46) and one with very high
severity (112). The colours in these plots are K-means' own numbering, which is **random** from one run to the next.

In [ ]:
def kmeans_fit(table, features, standardize=False, random_state=2206):
    X = table[features].to_numpy(dtype=float)
    if standardize:
        X = preprocessing.StandardScaler().fit_transform(X)
    km = KMeans(n_clusters=3, random_state=random_state, n_init=10).fit(X)
    return km.labels_, km.cluster_centers_


fig, ax = plt.subplots(2, 3, figsize=(15, 9))
for row, leaf_id in enumerate((46, 112)):
    img = cv2.medianBlur(read_rgb(leaf_id), 11)
    table = pixel_table(img)
    ax[row, 0].imshow(img); ax[row, 0].set_title(f"Leaf {leaf_id}")
    for col, (features, name) in enumerate([(RGB, "RGB"), (RGB + DIFFS, "RGB + differences")], start=1):
        labels, _ = kmeans_fit(table, features)
        ax[row, col].imshow(labels.reshape(img.shape[:2])); ax[row, col].set_title(f"K-means: {name}")
for a in ax.ravel(): a.axis("off")
plt.show()

In both leaves the lesion group follows the spots, including in the severe one. With the differences the lesion group
has fewer stray fragments along the vein of the leaf with few lesions.

## 6. Number the groups the same way in every image

K-means numbers its groups at random, so "group 0" is a different tissue in each image. My solution sorts the
groups by **the sum of the values of their centre** and renumbers them 0, 1, 2 in that order.

> Caution: the sum of the centre is a proxy for brightness only when the
> features are raw colours. With standardised features, and with the differences included, the sum mixes brightness
> with colour differences, and the order is not guaranteed to be background / healthy / lesion, and nothing checks
> it automatically. `describe_clusters` below prints the mean *raw* colour of each ordered group so that you can check it by eye.

In [ ]:
def order_clusters(labels, centers):
    """Renumber the groups by increasing sum of their centre coordinates."""
    order = np.argsort(centers.sum(axis=1))
    remap = {old: new for new, old in enumerate(order)}
    return np.vectorize(remap.get)(labels)


def describe_clusters(img_rgb, ordered_labels):
    """Mean raw RGB colour and share of pixels of each ordered group (to check which one is the lesion)."""
    flat = img_rgb.reshape(-1, 3)
    lab = ordered_labels.ravel()
    return pd.DataFrame({g: {"red": flat[lab == g, 0].mean(), "green": flat[lab == g, 1].mean(),
                             "blue": flat[lab == g, 2].mean(), "share_%": 100 * np.mean(lab == g)}
                         for g in np.unique(lab)}).T.round(1)


img = cv2.medianBlur(read_rgb(112), 11)
table = pixel_table(img)
labels, centers = kmeans_fit(table, RGB + DIFFS, standardize=True)
ordered = order_clusters(labels, centers)

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].imshow(labels.reshape(img.shape[:2])); ax[0].set_title("Random numbering")
ax[1].imshow(ordered.reshape(img.shape[:2]), cmap="nipy_spectral"); ax[1].set_title("Ordered by sum of centres")
for a in ax: a.axis("off")
plt.show()
print(describe_clusters(img, ordered))

## 7. Severity from the groups

Severity = lesion pixels ÷ (lesion + healthy pixels); the background group is left out.

**Which group is the lesion.** With raw features the lesion is group 0 and healthy tissue group 1. With
standardised features I set the opposite (lesion 1, healthy 0), after looking at the result. This shows how fragile
the ordering is; the same problem came back with the branch scans of version 2.0 (notebook 04, section 2).

I checked five leaves one by one (46, 112, 100, 150 and 151) with both settings.

In [ ]:
def severity_kmeans(leaf_id, standardize, use_diffs=True, lesion_label=None, healthy_label=None, show=False):
    """Severity (%) of one leaf. Default groups:
    raw features -> lesion 0 / healthy 1; standardised features -> lesion 1 / healthy 0."""
    if lesion_label is None:
        lesion_label, healthy_label = (1, 0) if standardize else (0, 1)
    img = cv2.medianBlur(read_rgb(leaf_id), 11)
    table = pixel_table(img)
    labels, centers = kmeans_fit(table, RGB + (DIFFS if use_diffs else []), standardize=standardize)
    ordered = order_clusters(labels, centers).reshape(img.shape[:2])
    lesion, healthy = np.sum(ordered == lesion_label), np.sum(ordered == healthy_label)
    if show:
        plt.figure(figsize=(5, 5)); plt.imshow(ordered, cmap="Oranges_r"); plt.axis("off"); plt.title(f"Leaf {leaf_id}")
        plt.show()
        print(describe_clusters(img, ordered))
    return 100 * lesion / (lesion + healthy)


for leaf_id in (46, 112, 100, 150, 151):
    print(leaf_id,
          "| raw:", round(severity_kmeans(leaf_id, standardize=False), 2),
          "| standardised:", round(severity_kmeans(leaf_id, standardize=True), 2),
          "| Pliman:", REFERENCE.loc[leaf_id])

## 8. Model 1 (raw features) against model 2 (standardised features)

I ran both on the 50 leaves. On most leaves they agree, but on several (for example 20, 25, 139, 142, 150, 151, 157)
model 1 gave between 63 % and 91 % where model 2 gave less than 15 %. Model 1 segmented 39 of the 50 leaves
correctly and failed on 11, all of them **very mature, dark-green leaves**. Standardising the features fixed them (for
leaf 155, 49.6 % without it and 7.1 % with it). It has a side effect: model 2 marks bigger lesions, because it is more
sensitive to the yellowish halos.

I did not test why standardising helps. A likely reason is that K-means works with distances, and raw colour values
and colour differences are on very different scales.

The severities of both models on the 50 leaves are in `data/standardization_comparison_50_leaves.csv`, so this part
runs without images.

In [ ]:
comparison = pd.read_csv("../../data/standardization_comparison_50_leaves.csv").set_index("fig_id")
comparison["model1_minus_model2"] = comparison["sev_model1_raw"] - comparison["sev_model2_standardised"]

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(comparison["sev_model1_raw"], comparison["sev_model2_standardised"])
for leaf_id, row in comparison.iterrows():
    ax.annotate(str(leaf_id), (row["sev_model1_raw"], row["sev_model2_standardised"]), fontsize=7)
lim = comparison[["sev_model1_raw", "sev_model2_standardised"]].to_numpy().max()
ax.plot([0, lim], [0, lim], "g--"); ax.plot([2, lim + 2], [0, lim], "r--"); ax.plot([0, lim], [2, lim + 2], "r--")
ax.set(xlabel="Model 1: raw features (%)", ylabel="Model 2: standardised features (%)")
plt.show()

disagree = comparison[comparison["model1_minus_model2"].abs() > 15].sort_values("model1_minus_model2")
print(f"{len(disagree)} leaves differ by more than 15 points:")
print(disagree[["sev_model1_raw", "sev_model2_standardised"]])

Next: [`03_knn_dead_end.ipynb`](03_knn_dead_end.ipynb) — could a supervised model take over from K-means? ·
Then the 2.0 stage: [`04_v2_branch_scans.ipynb`](04_v2_branch_scans.ipynb).